# Inference Demo — Loading and Using the Packaged Model

**Course:** Machine Learning Operations (MLOps) — AIMLCZG523, BITS Pilani WILP M.Tech. AIML
**Assignment:** Assignment 01
**Relates to:** Task 4 (Model Packaging & Reproducibility) and Task 6 (Model
Containerization) — this notebook demonstrates loading the exact artifact that
`api/main.py` loads in the deployed FastAPI service, proving the packaged pipeline is
fully self-contained and reusable outside of the training environment.

## Objective
Load `models/heart_disease_pipeline.joblib` (produced by `python src/train.py`) in a
brand-new Python session with NO access to the original training code path for
preprocessing, and run predictions on a few sample patient records — exactly mirroring
what happens inside the Docker container / Kubernetes pod at request time.


In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import os
os.chdir("..")

MODEL_PATH = "models/heart_disease_pipeline.joblib"
pipeline = joblib.load(MODEL_PATH)
print(f"Loaded pipeline from {MODEL_PATH}")
print(pipeline)


Loaded pipeline from models/heart_disease_pipeline.joblib
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                 

## Sample Patients

Three illustrative records: one from the raw dataset (known label), and two synthetic
edge cases -- a low-risk profile and a high-risk profile -- to sanity-check the model's
directional behaviour.


In [2]:
samples = pd.DataFrame([
    {   # Real record from the dataset (target = 0, no disease)
        "age": 63, "sex": 1, "cp": 1, "trestbps": 145, "chol": 233,
        "fbs": 1, "restecg": 2, "thalach": 150, "exang": 0,
        "oldpeak": 2.3, "slope": 3, "ca": 0, "thal": 6,
    },
    {   # Synthetic low-risk profile: young, high max heart rate, no angina, low oldpeak
        "age": 35, "sex": 0, "cp": 4, "trestbps": 110, "chol": 180,
        "fbs": 0, "restecg": 0, "thalach": 185, "exang": 0,
        "oldpeak": 0.2, "slope": 1, "ca": 0, "thal": 3,
    },
    {   # Synthetic high-risk profile: older, typical angina, low max HR, high oldpeak
        "age": 68, "sex": 1, "cp": 4, "trestbps": 160, "chol": 290,
        "fbs": 1, "restecg": 2, "thalach": 108, "exang": 1,
        "oldpeak": 3.8, "slope": 2, "ca": 3, "thal": 7,
    },
])
samples


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0,6
1,35,0,4,110,180,0,0,185,0,0.2,1,0,3
2,68,1,4,160,290,1,2,108,1,3.8,2,3,7


In [3]:
predictions = pipeline.predict(samples)
probabilities = pipeline.predict_proba(samples)

results = samples.copy()
results["prediction"] = predictions
results["prediction_label"] = results["prediction"].map({0: "No Disease", 1: "Disease Present"})
results["confidence"] = [proba[pred] for pred, proba in zip(predictions, probabilities)]
results["probability_disease"] = probabilities[:, 1]

results[["age", "sex", "cp", "prediction_label", "confidence", "probability_disease"]]


,age,sex,cp,prediction_label,confidence,probability_disease
0,63,1,1,No Disease,0.834465,0.165535
1,35,0,4,No Disease,0.977692,0.022308
2,68,1,4,Disease Present,0.997341,0.997341


**Observations:** the low-risk synthetic profile (young, high exercise capacity, no
angina) is predicted as no-disease with high confidence, and the high-risk profile
(older, angina present, low exercise heart rate, high ST depression, more blocked
vessels) is predicted as disease-present — directionally consistent with the EDA
findings (`notebooks/01_eda.ipynb`) and clinical domain knowledge, which is a useful
sanity check beyond pure held-out-set metrics.


## Equivalent REST API Call

The exact same pipeline, served by `api/main.py` (Docker container / Kubernetes pod),
is called like this from the command line:

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
        "age": 63, "sex": 1, "cp": 1, "trestbps": 145, "chol": 233,
        "fbs": 1, "restecg": 2, "thalach": 150, "exang": 0,
        "oldpeak": 2.3, "slope": 3, "ca": 0, "thal": 6
      }'
```

Expected response shape:
```json
{
  "prediction": 0,
  "prediction_label": "No Heart Disease",
  "confidence": 0.8345,
  "probability_disease": 0.1655
}
```
